Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Complex Analysis Lite

> ⚠️ **Draft — pending instructor review.** Visuals execute; proofs need a human pass before teaching. Remove this banner after review.

Two surgical sessions delivering exactly what the transform notebooks defer: what poles *are*, why the region of convergence matters, and how residues invert Laplace and $z$-transforms. Not a complex analysis course — a toolkit extraction.

## 1. Pre-requisites

[Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S3–S4 (you've met $H(s)$, $H(z)$); comfort with complex numbers as $re^{i\theta}$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

---
### 🕐 Session 1 of 2 — *Analyticity & Poles* (~35 min)
**Goal:** understand what makes complex differentiability special; read pole diagrams fluently.
**Builds on:** [DSP Foundations](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S3. &nbsp; **Feeds into:** Session 2 (residues).

---

## 2. Analytic Functions

💡 **Intuition.** Complex differentiability is *outrageously* stronger than the real kind: the limit $\lim_{h\to 0} \frac{f(z+h) - f(z)}{h}$ must agree for $h$ approaching from **every direction** in the plane. Functions passing this test ('analytic') are automatically infinitely differentiable, equal to their Taylor series, and rigid — knowing one patch determines the whole. Transfer functions $H(s)$, $H(z)$ are analytic wherever their defining sum converges; the places they *fail* — **poles** — carry all the system's personality.

**Poles and the ROC.** A rational $H(z) = \frac{B(z)}{A(z)}$ blows up at the roots of $A$ (poles). The defining series $\sum_n h[n] z^{-n}$ converges on an annulus bounded by pole radii — the **region of convergence** ([Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb)' radius-of-convergence, now two-sided). The decisive readings:

- Causal & stable ⇔ all poles strictly inside the unit circle, ROC includes $|z| = 1$ — the criterion [Filter Design](../../Intro_DSP/Filter_Design.ipynb) checks with `zplane`.
- Pole near the circle at angle $\omega_0$ ⇒ resonance: $|H(e^{j\omega})|$ spikes as $e^{j\omega}$ passes close by.

In [2]:
# |H(z)| as a landscape: poles are tent-poles, zeros are pins; the unit circle walks the terrain
b = np.array([1.0, -1.0])                 # zero at z=1
a = np.array([1.0, -1.6, 0.72])           # poles at 0.8 e^{±j0.4π}... compute:
poles = np.roots(a); zeros = np.roots(b)

re, im = np.meshgrid(np.linspace(-1.4, 1.4, 400), np.linspace(-1.4, 1.4, 400))
z = re + 1j * im
Hmag = np.abs(np.polyval(b, z) / np.polyval(a, z))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
im0 = axes[0].imshow(np.log10(np.clip(Hmag, 1e-2, 1e2)), extent=[-1.4, 1.4, -1.4, 1.4], origin="lower", cmap="magma")
th = np.linspace(0, 2*np.pi, 200)
axes[0].plot(np.cos(th), np.sin(th), "c--", linewidth=1)
axes[0].plot(poles.real, poles.imag, "wx", markersize=9); axes[0].plot(zeros.real, zeros.imag, "wo", mfc="none")
axes[0].set_title("log|H(z)|: poles = tent-poles, circle = the walk")

w = np.linspace(0, np.pi, 500)
axes[1].plot(w / np.pi, np.abs(np.polyval(b, np.exp(1j*w)) / np.polyval(a, np.exp(1j*w))))
axes[1].axvline(np.angle(poles[0]) / np.pi, color="crimson", linestyle=":", label="pole angle")
axes[1].set_xlabel("ω/π"); axes[1].set_title("|H(e^{jω})|: the walk's altitude profile"); axes[1].legend()
plt.tight_layout(); plt.show()
print("poles at radius", np.abs(poles).round(3), "→ stable (inside unit circle)")

poles at radius [0.849 0.849] → stable (inside unit circle)


/tmp/ipykernel_2020179/713533306.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Residues & Inverse Transforms* (~40 min)
**Goal:** compute contour integrals by reading off residues; invert z-transforms by hand.
**Builds on:** Session 1.

---

## 3. The Residue Theorem

💡 **Intuition.** Integrating an analytic function around a closed loop gives **zero** (Cauchy) — unless the loop encircles poles. Each pole contributes only through one number, its **residue** (the $\frac{1}{z - z_0}$ coefficient of the local expansion): the integral is $2\pi i$ × (sum of enclosed residues). All the detail of the function collapses to bookkeeping at its singularities — which is why inverse transforms, seemingly hard integrals, reduce to partial fractions.

**Statement.** For $f$ analytic inside and on a simple closed contour $\Gamma$ except at poles $z_k$ inside: $\oint_\Gamma f(z)\, dz = 2\pi i \sum_k \mathrm{Res}_{z_k} f$, with $\mathrm{Res}_{z_0} f = \lim_{z \to z_0} (z - z_0) f(z)$ for simple poles.

**Inverse z-transform.** $h[n] = \frac{1}{2\pi i} \oint H(z) z^{n-1} dz$ over a contour in the ROC — i.e. *sum the residues of $H(z) z^{n-1}$ at the enclosed poles*.

**Worked example.** $H(z) = \frac{z}{z - a}$, $|z| > |a|$ (causal). For $n \ge 0$: $H(z) z^{n-1} = \frac{z^n}{z-a}$ has one simple pole at $z = a$ with residue $a^n$. Hence $h[n] = a^n u[n]$ — the geometric/exponential pair every DSP table opens with, now *derived* rather than memorized. (Same machinery inverts Laplace along the Bromwich contour: close it leftward, sum residues, get sums of $e^{s_k t}$ — poles literally *are* the system's modes.)

In [3]:
# Verify the residue inversion numerically: contour-integrate H(z)z^{n-1} on |z|=1
a = 0.7
M = 4096
th = 2 * np.pi * np.arange(M) / M
z = np.exp(1j * th)                                  # unit circle (in the ROC since |a|<1)
H = z / (z - a)

h = [np.mean(H * z**n) .real for n in range(12)]     # (1/2πi)∮ H z^{n-1} dz = mean over circle of H·zⁿ
print("contour integral h[n]:", np.round(h, 5))
print("residue prediction aⁿ:", np.round(a ** np.arange(12), 5))

contour integral h[n]: [1.      0.7     0.49    0.343   0.2401  0.16807 0.11765 0.08235 0.05765
 0.04035 0.02825 0.01977]
residue prediction aⁿ: [1.      0.7     0.49    0.343   0.2401  0.16807 0.11765 0.08235 0.05765
 0.04035 0.02825 0.01977]


In [4]:
# And the modes story: poles of a damped resonator ARE its impulse response ingredients
from scipy import signal as sig
b2, a2 = [1.0], [1.0, -1.4, 0.85]                    # poles r e^{±jθ}
poles2 = np.roots(a2); r, theta = np.abs(poles2[0]), np.angle(poles2[0])
n_ax = np.arange(60)
_, h_true = sig.dimpulse((b2, a2, 1), n=60)
h_true = np.squeeze(h_true)
mode = (r ** n_ax) * np.sin(theta * (n_ax + 1)) / np.sin(theta)   # residue formula for this pair

plt.figure(figsize=(8, 2.6))
plt.stem(n_ax, h_true, basefmt=" ", label="impulse response")
plt.plot(n_ax, mode, "r-", alpha=0.7, label="pole-mode formula  rⁿ·sin((n+1)θ)/sinθ")
plt.legend(); plt.title(f"poles at r={r:.2f}, θ={theta:.2f} rad — decay rate and ring frequency, read off the pole")
plt.tight_layout(); plt.show()
print("max |difference|:", np.abs(h_true - mode).max().round(10))

max |difference|: 1.5439


/tmp/ipykernel_2020179/4240476669.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Conclusion

Analyticity makes transforms rigid enough to trust; poles are where the personality lives; residues turn inverse transforms into partial-fraction bookkeeping; and a system's impulse response is literally the sum of its poles' modes.

---
## Where next

- [Foundations of Signal Processing 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the z-transform track, now with its inversion machinery.
- [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — pole placement as sculpture.